# 🚀 CommunityLab — Laboratorio Interactivo de IA (Data Science)
**Equipo 39 · ONE G10 LATAM (Oracle Next Education & Alura)**  
**Data Scientist:** Danny de Jesús González Portillo  
**Línea Base Oficial:** Semana 3 Consolidada (`src.agentes.procesamiento`, `src.agentes.grafo`, `src.datos.ingesta`)

---

### 🎯 Propósito del Cuaderno Técnico:
Este cuaderno interactivo proporciona una validación integral, modular y visual del motor de IA de **CommunityLab**:

1. **Ingesta y Puntuación de Relevancia (`src.datos`):** Carga y limpieza de datos comunitarios con filtrado matemático.
2. **Grafo de Estados y Enrutamiento (`src.agentes.grafo`):** Despacho condicional hacia los canales de marketing y soporte.
3. **Procesamiento de Lotes y Resiliencia (`src.agentes.procesamiento`):** Ejecución estructurada con observabilidad y reintentos.
4. **Inferencia Cognitiva en Vivo (`src.agentes.cadenas`):** Extracción semántica estructurada bajo esquema Pydantic.
5. **Prompts Few-Shot y Contrato de Salida (`src.agentes.prompts`):** Estandarización de activos institucionales (v1.3).
6. **Integración Oficial NVIDIA NIM (`src.agentes.proveedores.nvidia_nim`):** Conexión con Nemotron-3.5-Lightning.
7. **Benchmark Empírico y Resiliencia Fallback:** Evaluación de casos límite ambiguos y tolerancia a fallos.


In [1]:
import os
import sys
import json
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

# 1. Cargar variables de entorno locales (.env)
load_dotenv(override=True)

# 2. Asegurar que la raíz del proyecto esté en sys.path
RAIZ_PROYECTO = Path(".").resolve()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.agentes.configuracion_ia import (
    obtener_proveedor_analisis,
    obtener_proveedor_generacion,
    MODELO_NVIDIA_NEMOTRON,
)
from src.agentes.modelo_ia import obtener_configuracion_modelo

nvidia_key = os.getenv("NVIDIA_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")
cohere_key = os.getenv("COHERE_API_KEY")

config_activa = obtener_configuracion_modelo()
prov_analisis = obtener_proveedor_analisis().upper()
prov_gen = obtener_proveedor_generacion().upper()

print("=" * 75)
print("✅ Entorno de CommunityLab Inicializado Correctamente (Semana 3 Consolidada)")
print("=" * 75)
print(f"📁 Raíz del proyecto      : {RAIZ_PROYECTO}")
print(f"⚡ Proveedor Análisis      : {prov_analisis} (NVIDIA NIM por defecto)")
print(f"🎨 Proveedor Generación    : {prov_gen}")
print(f"🧠 Modelo Activo Primario  : {config_activa.get('identificador_modelo')}")
print("🛡️ Fallback Automático     : Gemini (Activo ante 429, timeouts o fallos)")
print(f"🔑 NVIDIA_API_KEY          : {'✅ Configurada' if nvidia_key else '⚠️ Pendiente'}")
print(f"🔑 GEMINI_API_KEY          : {'✅ Configurada' if gemini_key else '⚠️ Pendiente'}")
print(f"🔑 COHERE_API_KEY          : {'✅ Configurada' if cohere_key else '⚠️ Pendiente'}")
print("=" * 75)


✅ Entorno de CommunityLab Inicializado Correctamente (Semana 3 Consolidada)
📁 Raíz del proyecto      : C:\Users\Danny\Documents\Proyecto de CommunityLab MotorInteligente\communitylab
⚡ Proveedor Análisis      : NVIDIA_NIM (NVIDIA NIM por defecto)
🎨 Proveedor Generación    : NVIDIA_NIM
🧠 Modelo Activo Primario  : nvidia/nemotron-3.5-lightning-30b-a3b
🛡️ Fallback Automático     : Gemini (Activo ante 429, timeouts o fallos)
🔑 NVIDIA_API_KEY          : ✅ Configurada
🔑 GEMINI_API_KEY          : ✅ Configurada
🔑 COHERE_API_KEY          : ✅ Configurada


---
## 📥 Paso 1: Ingesta de Datos y Scoring de Relevancia (`src.datos.ingesta`)
Validación del dataset maestro (`src/datos/mensajes_comunidad_simulados.json`), limpieza textual y segmentación:
- **`incluido_sentimiento`**: Todo mensaje con contenido sustancial participa en la métrica de salud comunitaria.
- **`elegible_contenido`**: Mensajes con puntuación >= 70 calificados para generación de activos institucionales.

In [2]:
from scripts.apoyo_demostraciones import cargar_paquete_demostracion, obtener_evaluaciones_por_id
from src.datos.ingesta import limpiar_texto

# 1. Cargar el paquete oficial preparado por Datos para el flujo de IA
paquete = cargar_paquete_demostracion(tamano_ciclo=12)
evaluaciones = obtener_evaluaciones_por_id(paquete['informe'])

resumen = paquete['informe']['resumen']
total_interacciones = resumen.get('total', len(paquete['estados']))
seleccionadas = resumen.get('seleccionadas')
elegibles = len(paquete['plan']['ids_contenido'])
ciclos = len(paquete['plan']['ciclos'])

print(f"📊 Total de interacciones procesadas: {total_interacciones}")
print(f"✅ Interacciones seleccionadas: {seleccionadas}")
print(f"🎯 Mensajes elegibles para activos de marketing: {elegibles}")
print(f"🔄 Ciclos planificados por lotes: {ciclos}")

# Mostrar una muestra de los mensajes evaluados
print("\n" + "-" * 70)
print("Muestra de Mensajes y sus Puntuaciones de Relevancia (DA):")
print("-" * 70)
for estado in paquete['estados'][:4]:
    msg_id = estado['id']
    autor = estado.get('autor', 'N/A')
    canal = estado.get('canal', 'N/A')
    score = estado.get('score_relevancia')
    tipo = estado.get('tipo_original')
    texto = estado.get('texto', '')[:85]
    elegible = '✅ SÍ' if msg_id in paquete['plan']['ids_contenido'] else '❌ NO'
    print(f"🔹 ID: {msg_id} | Autor: {autor} | Canal: {canal}")
    print(f"   Score: {score}/100 | Elegible Activos: {elegible} | Tipo: {tipo}")
    print(f"   Texto limpio: \"{texto}...\"\n")

📊 Total de interacciones procesadas: 23
✅ Interacciones seleccionadas: 19
🎯 Mensajes elegibles para activos de marketing: 19
🔄 Ciclos planificados por lotes: 2

----------------------------------------------------------------------
Muestra de Mensajes y sus Puntuaciones de Relevancia (DA):
----------------------------------------------------------------------
🔹 ID: int-001 | Autor: Camila Restrepo | Canal: #logros-y-empleos
   Score: 90/100 | Elegible Activos: ✅ SÍ | Tipo: testimonio
   Texto limpio: "Comunidad, gracias a este programa conseguí mi primer trabajo como analista de datos ..."

🔹 ID: int-002 | Autor: Diego Fernández | Canal: #dudas-langgraph
   Score: 80/100 | Elegible Activos: ✅ SÍ | Tipo: pregunta_tecnica
   Texto limpio: "Tengo dudas sobre cómo estructurar los nodos condicionales en LangGraph cuando la res..."

🔹 ID: int-003 | Autor: Andrés Kuri | Canal: #logros-y-empleos
   Score: 75/100 | Elegible Activos: ✅ SÍ | Tipo: testimonio
   Texto limpio: "Pasé de no saber pro

---
## 🔀 Paso 2: Grafo de LangGraph y Enrutamiento Determinista (`src.agentes.grafo`)
Evaluación del enrutador condicional sobre estados analizados para derivar activos:
- `linkedin` / `caso_exito`: Testimonios destacados y logros profesionales de alto impacto.
- `preguntas_frecuentes`: Dudas técnicas o de programa recurrentes.
- `boletin`: Novedades formativas y convocatorias.
- `insight_mejora`: Oportunidades de optimización y retroalimentación crítica del curso.

In [3]:
from src.agentes.modelos import AnalisisMensaje
from src.agentes.estado_agente import EstadoAgente
from src.agentes.nodos.nodo_enrutador import determinar_rutas

# 1. Definimos dos estados simulados con su análisis para probar el enrutador
estado_testimonio: EstadoAgente = {
    'id': 'int-022',
    'texto': 'Comunidad, quedé seleccionada como Desarrolladora Junior de IA!',
    'sentimiento': 'muy_positivo',
    'tema_principal': 'empleabilidad',
    'subtema': 'primer_empleo',
    'tipo_detectado': 'testimonio',
    'elegible_contenido': True,
    'score_relevancia': 95,
}

estado_duda_tecnica: EstadoAgente = {
    'id': 'int-002',
    'texto': 'Tengo dudas sobre cómo estructurar los nodos condicionales en LangGraph.',
    'sentimiento': 'neutral',
    'tema_principal': 'datos_ia',
    'subtema': 'langgraph',
    'tipo_detectado': 'pregunta_tecnica',
    'elegible_contenido': True,
    'score_relevancia': 80,
}

# 2. Evaluar las rutas decididas por el enrutador para cada caso
print('🎯 EVALUACIÓN DE ENRUTAMIENTO CONDICIONAL (LangGraph Router):')
print('-' * 70)
for caso in [estado_testimonio, estado_duda_tecnica]:
    res = determinar_rutas(caso)
    print(f"📌 Caso [{caso['id']}] - Tipo: '{caso['tipo_detectado']}' | Sentimiento: '{caso['sentimiento']}'")
    print(f"   👉 Rutas asignadas por el grafo: {res['rutas']}\n")

🎯 EVALUACIÓN DE ENRUTAMIENTO CONDICIONAL (LangGraph Router):
----------------------------------------------------------------------
📌 Caso [int-022] - Tipo: 'testimonio' | Sentimiento: 'muy_positivo'
   👉 Rutas asignadas por el grafo: ['caso_exito', 'boletin', 'linkedin']

📌 Caso [int-002] - Tipo: 'pregunta_tecnica' | Sentimiento: 'neutral'
   👉 Rutas asignadas por el grafo: ['preguntas_frecuentes']



---
## ⚙️ Paso 3: Procesamiento por Lotes y Entrega (`src.agentes.procesamiento`)
Ejecución del pipeline en bloque mediante `procesar_estados_por_lotes`:
- Segmentación en ciclos planificados con métricas de latencia por nodo.
- Trazabilidad de observabilidad y aislamiento de excepciones transitorias.

---
## 🧠 Paso 4: Inferencia Cognitiva en Vivo (NVIDIA NIM con Fallback a Gemini)
Ejecución en tiempo real de la cadena de análisis (`cadena_analisis`) con observabilidad acoplada (`config_ejecucion`).
El modelo extrae el esquema estructurado `AnalisisMensaje` (sentimiento, tema, subtema y tipo de interacción).

In [4]:
from src.agentes.cadenas import cadena_analisis
from src.agentes.modelos import AnalisisMensaje
from src.agentes.nodos.nodo_enrutador import determinar_rutas
from src.agentes.estado_agente import EstadoAgente
from src.agentes.observabilidad import config_ejecucion

# 1. Definir dos casos reales del proyecto (Testimonio de Empleo y Duda Técnica)
casos_prueba = [
    {
        "id": "int-022",
        "autor": "Mariana Souza",
        "canal": "#logros-y-empleos",
        "origen": "Discord",
        "tipo_original": "testimonio",
        "score_relevancia": 95,
        "elegible_contenido": True,
        "texto": "Comunidad, gracias al curso y al proyecto de IA de mi portfolio quedé seleccionada como Desarrolladora Junior de IA! Muy agradecida!"
    },
    {
        "id": "int-002",
        "autor": "Diego Fernández",
        "canal": "#dudas-langgraph",
        "origen": "Discord",
        "tipo_original": "pregunta_tecnica",
        "score_relevancia": 80,
        "elegible_contenido": True,
        "texto": "Tengo dudas sobre cómo estructurar los nodos condicionales en LangGraph cuando la respuesta del LLM necesita reintento. ¿Alguien tiene un ejemplo práctico?"
    }
]

print("🚀 PROCESANDO CASOS CON LA CADENA OFICIAL DE ANÁLISIS (NVIDIA NIM + FALLBACK GEMINI):")
print("=" * 75)
for caso in casos_prueba:
    print(f"\n📩 Analizando caso [{caso['id']}] de '{caso['autor']}'...")
    
    # Invocamos la cadena oficial pasando intento=1 requerido por config_ejecucion
    config_obs = config_ejecucion("analisis_demostracion", intento=1, id_interaccion=caso["id"])
    analisis: AnalisisMensaje = cadena_analisis.invoke({
        "canal": caso["canal"],
        "idioma": "es",
        "origen": caso["origen"],
        "tipo_original": caso["tipo_original"],
        "texto": caso["texto"]
    }, config=config_obs)
    
    print(f"   ✅ Sentimiento     : {analisis.sentimiento}")
    print(f"   ✅ Tema principal  : {analisis.tema_principal} ({analisis.subtema})")
    print(f"   ✅ Tipo detectado  : {analisis.tipo_detectado}")
    
    # Construimos el estado para el enrutador determinista
    estado_evaluado: EstadoAgente = {
        "id": caso["id"],
        "texto": caso["texto"],
        "sentimiento": analisis.sentimiento,
        "tema_principal": analisis.tema_principal,
        "subtema": analisis.subtema,
        "tipo_detectado": analisis.tipo_detectado,
        "elegible_contenido": caso["elegible_contenido"],
        "rutas": [],
        "activos_generados": {}
    }
    
    rutas = determinar_rutas(estado_evaluado)
    print(f"   🔀 Rutas asignadas : {rutas}")

print("=" * 75)
print("🎉 ¡Análisis y enrutamiento en vivo completados con éxito!")


🚀 PROCESANDO CASOS CON LA CADENA OFICIAL DE ANÁLISIS (NVIDIA NIM + FALLBACK GEMINI):

📩 Analizando caso [int-022] de 'Mariana Souza'...
   ✅ Sentimiento     : muy_positivo
   ✅ Tema principal  : empleabilidad (selección como Desarrolladora Junior)
   ✅ Tipo detectado  : testimonio
   🔀 Rutas asignadas : {'rutas': ['caso_exito', 'boletin', 'linkedin']}

📩 Analizando caso [int-002] de 'Diego Fernández'...
   ✅ Sentimiento     : neutral
   ✅ Tema principal  : programacion (nodos condicionales LangGraph)
   ✅ Tipo detectado  : pregunta_tecnica
   🔀 Rutas asignadas : {'rutas': ['preguntas_frecuentes']}
🎉 ¡Análisis y enrutamiento en vivo completados con éxito!


In [5]:
from scripts.apoyo_demostraciones import cargar_paquete_demostracion
from src.agentes.grafo import procesar_estados_por_lotes

# 1. Cargar el paquete completo de demostración preparado por Datos
paquete_demostracion = cargar_paquete_demostracion(tamano_ciclo=12)

print(f"📦 Paquete cargado:")
print(f"   • Total estados: {len(paquete_demostracion['estados'])}")
print(f"   • Mensajes en plan de contenido: {len(paquete_demostracion['plan']['ids_contenido'])}")
print(f"   • Ciclos planificados: {len(paquete_demostracion['plan']['ciclos'])}")

# 2. Probamos una corrida controlada sobre 2 estados seleccionados
ids_seleccionados = ["int-022", "int-002"]
estados_muestra = [
    e for e in paquete_demostracion["estados"] 
    if e["id"] in ids_seleccionados
]
ids_contenido = set(paquete_demostracion["plan"]["ids_contenido"])

print("\n🚀 Ejecutando `procesar_estados_por_lotes` sobre los estados seleccionados...")
resultados_lote = procesar_estados_por_lotes(estados_muestra, ids_contenido=ids_contenido)

print(f"✅ Procesamiento finalizado. Total resultados obtenidos: {len(resultados_lote)}")
for res in resultados_lote:
    print(f"\n🔹 Resultado ID: {res['id']}")
    print(f"   Sentimiento: {res.get('sentimiento')} | Tema: {res.get('tema_principal')}")
    print(f"   Rutas activadas: {res.get('rutas')}")
    print(f"   Activos generados: {list(res.get('activos_generados', {}).keys())}")
    if res.get('errores'):
        print(f"   ⚠️ Nota/Errores: {res['errores']}")

📦 Paquete cargado:
   • Total estados: 23
   • Mensajes en plan de contenido: 19
   • Ciclos planificados: 2

🚀 Ejecutando `procesar_estados_por_lotes` sobre los estados seleccionados...
✅ Procesamiento finalizado. Total resultados obtenidos: 2

🔹 Resultado ID: int-002
   Sentimiento: neutral | Tema: programacion
   Rutas activadas: ['preguntas_frecuentes']
   Activos generados: ['preguntas_frecuentes']

🔹 Resultado ID: int-022
   Sentimiento: muy_positivo | Tema: empleabilidad
   Rutas activadas: ['caso_exito', 'boletin', 'linkedin']
   Activos generados: ['caso_exito', 'boletin', 'linkedin']


---
## 📋 Paso 5: Contrato de Salida DS y Plantillas Few-Shot (`src.agentes.prompts`)
Inspección del contrato unificado de Ciencia de Datos (v1.3) y las directrices de tono institucional para los redactores de contenido.

In [6]:
from src.agentes.prompts.linkedin import prompt_linkedin
from src.agentes.prompts.preguntas_frecuentes import prompt_preguntas_frecuentes
from src.agentes.contrato_salida_ciencia_datos import VERSION_CONTRATO_SALIDA_DS

print('=' * 75)
print(f'📋 CONTRATO DE SALIDA DE DATA SCIENCE: Versión {VERSION_CONTRATO_SALIDA_DS}')
print('=' * 75)

print('\n📄 Vista previa de la plantilla Few-Shot para LinkedIn (prompt_linkedin):')
print('-' * 75)
print(prompt_linkedin.messages[0].prompt.template[:420] + '\n... [TRUNCADO PARA VISUALIZACIÓN]')

print('\n❓ Vista previa de la plantilla para FAQs (prompt_preguntas_frecuentes):')
print('-' * 75)
print(prompt_preguntas_frecuentes.messages[0].prompt.template[:400] + '\n... [TRUNCADO PARA VISUALIZACIÓN]')


📋 CONTRATO DE SALIDA DE DATA SCIENCE: Versión 1.3

📄 Vista previa de la plantilla Few-Shot para LinkedIn (prompt_linkedin):
---------------------------------------------------------------------------

Eres redactor del equipo de Marketing y Community Management
de una comunidad educativa y tecnológica.

Tu tarea es transformar un testimonio real de un miembro de la
comunidad en una publicación para el LinkedIn oficial de la organización.

La publicación debe estar escrita desde una voz institucional.
Nunca escribas como si fueras la persona autora del testimonio.

# OBJETIVO

Convertir una experiencia real de la 
... [TRUNCADO PARA VISUALIZACIÓN]

❓ Vista previa de la plantilla para FAQs (prompt_preguntas_frecuentes):
---------------------------------------------------------------------------

Eres redactor de preguntas frecuentes para una comunidad educativa
y tecnológica.

Tu tarea es transformar una pregunta real de la comunidad en una
entrada breve y útil para una sección de pregun

---
## ⚡ Paso 6: Verificación Directa de NVIDIA NIM (Nemotron-3.5-Lightning)
Invocación directa del endpoint de inferencia NVIDIA NIM bajo el estándar OpenAI-compatible:
- **Modelo Primario:** `nvidia/nemotron-3.5-lightning-30b-a3b` a temperatura 0.
- **Contrato:** Validación estricta contra el esquema Pydantic oficial del proyecto.

In [7]:
from langchain_core.messages import HumanMessage
from src.agentes.proveedores.nvidia_nim import obtener_configuracion_nvidia
from src.agentes.configuracion_ia import MODELO_NVIDIA_NEMOTRON
from src.agentes.modelo_ia import (
    obtener_modelo_analisis_estructurado,
    obtener_modelo_generacion_estructurado,
    invocar_modelo_con_fallback,
)
from src.agentes.modelos import AnalisisMensaje, PublicacionLinkedIn
from src.agentes.fallback_ia import invocar_modelo_con_fallback

config_nv = obtener_configuracion_nvidia()
print("=" * 75)
print("🚀 MÓDULO OFICIAL DE NVIDIA NIM Y ARQUITECTURA DE FALLBACK")
print("=" * 75)
print(f"• Proveedor primario   : {config_nv['proveedor']}")
print(f"• Identificador modelo : {config_nv['identificador_modelo']}")
print(f"• Base URL oficial     : {config_nv['base_url']}")

# 1. Obtener directamente el modelo estructurado de NIM
modelo_nim = obtener_modelo_analisis_estructurado(AnalisisMensaje, proveedor="nvidia_nim")

print("\n🧪 Invocando modelo NIM estructurado directamente...")
# El adaptador de NIM recibe una lista de mensajes de LangChain (HumanMessage) o un prompt formateado
mensajes_entrada = [
    HumanMessage(content="Excelente curso de Machine Learning, me ayudó a conseguir trabajo como Data Scientist!")
]
resultado = modelo_nim.invoke(mensajes_entrada)
print(f"   ✅ Sentimiento    : {resultado.sentimiento}")
print(f"   ✅ Tema principal : {resultado.tema_principal}")
print(f"   ✅ Subtema        : {resultado.subtema}")
print(f"   ✅ Tipo detectado : {resultado.tipo_detectado}")
print("\n🎉 ¡NVIDIA NIM responde bajo el contrato oficial de arquitectura del proyecto!")


🚀 MÓDULO OFICIAL DE NVIDIA NIM Y ARQUITECTURA DE FALLBACK
• Proveedor primario   : NVIDIA NIM
• Identificador modelo : nvidia/nemotron-3.5-lightning-30b-a3b
• Base URL oficial     : https://integrate.api.nvidia.com/v1

🧪 Invocando modelo NIM estructurado directamente...
   ✅ Sentimiento    : muy_positivo
   ✅ Tema principal : empleabilidad
   ✅ Subtema        : conseguir trabajo
   ✅ Tipo detectado : testimonio

🎉 ¡NVIDIA NIM responde bajo el contrato oficial de arquitectura del proyecto!


---
## 🔬 Paso 7: Aporte DS — Benchmark Empírico de Casos Límite Ambiguos
Herramienta automatizada (`scripts/benchmark_calibracion_ambiguos.py`) desarrollada por Danny González para someter los casos límite debatibles (`int-006`, `int-011`, `int-016`, `int-020`) a evaluación comparativa entre **NVIDIA Nemotron** y **Google Gemini**.

In [ ]:
# Visualización estructurada de los resultados del benchmark empírico
import json
from pathlib import Path

ruta_benchmark = Path("docs/benchmark_calibracion_casos_ambiguos.json")
if ruta_benchmark.exists():
    with open(ruta_benchmark, "r", encoding="utf-8") as f:
        datos_bench = json.load(f)

    # Compatibilidad: si el JSON tiene la nueva metadata (Punto 21), extraer 'resultados'
    casos = datos_bench.get("resultados", datos_bench) if isinstance(datos_bench, dict) else datos_bench

    print("=" * 80)
    print("🔬 RESULTADOS DEL BENCHMARK EMPÍRICO MULTI-LLM (DATA SCIENCE)")
    print(f"Fuente oficial: {ruta_benchmark}")
    print("=" * 80)
    for caso in casos:
        print(f"\n📩 [{caso['id']}] {caso['autor']}: \"{caso['texto'][:55]}...\"")
        nim = caso.get("evaluacion_nim") or {}
        gem = caso.get("evaluacion_gemini") or {}
        if "error" in nim:
            print(f"   ❌ NVIDIA NIM   : Error ({nim['error']})")
        else:
            print(f"   🟢 NVIDIA NIM   : {nim.get('sentimiento')} | {nim.get('tema_principal')} | {nim.get('tipo_detectado')}")
        if "error" in gem:
            print(f"   ❌ Google Gemini: Error ({gem['error']})")
        else:
            print(f"   🔵 Google Gemini: {gem.get('sentimiento')} | {gem.get('tema_principal')} | {gem.get('tipo_detectado')}")
else:
    print(f"El archivo {ruta_benchmark} no existe aún. Ejecuta scripts/benchmark_calibracion_ambiguos.py primero.")


---
## 🎯 Resumen de Estado del Equipo y Aportes de Alto Impacto

### 📍 Estado Actual del Proyecto y Sincronización del Equipo 39:
- **Línea base integrada:** `develop` tiene consolidada la suite de Data Science (PR #22) con 152 tests en verde.
- **Adopción Oficial de NVIDIA NIM con Fallback a Gemini (Rama `feature/DS-Semana3-Nvidia`):**
  - Arnold Bustinza (DS) lideró la integración del adaptador base (commit `6e6e0a6`).
  - Quedaron en verde las 46 pruebas de aceptación de Data Analyst (`test_adaptador_nvidia_integracion` y `test_respaldo_nim_gemini`).
  - Documentación asociada: `docs/manual_lectura_codigo_nvidia.md` y `docs/guia_pruebas_nvidia.md`.

### 💡 Aportes Proactivos de Alto Impacto de Danny (Data Scientist — PR #27):
1. **⚡ Suite de Benchmark Empírico Multi-Modelo (`scripts/benchmark_calibracion_ambiguos.py`):**
   - Evaluación automatizada de calibración semántica sobre los 4 casos límite del dataset (`int-006`, `int-011`, `int-016`, `int-020`).
   - Generación del reporte de resultados en `docs/benchmark_calibracion_casos_ambiguos.json` y guía técnica en `docs/guia_benchmark_y_resiliencia_fallback.md`.
2. **🛡️ Resiliencia de Fallback ante Estados DEGRADED (`src/agentes/fallback_ia.py`):**
   - Identificación y tratamiento del error HTTP 400 por función degradada en NVIDIA NIM, incorporando la regla `servicio_degradado` para conmutar automáticamente a Gemini.
   - Incorporación de prueba unitaria dedicada en `tests/agents/test_fallback_analisis.py` con 0 regresiones.
3. **🤝 Validación Cruzada con Data Analyst (Gustavo / Jhonattan):**
   - Verificación del comportamiento del caso `int-020`, respaldando que la columna `tipo_original: "feedback"` de DA se preserve como ancla inmutable en curaduría.
4. **🚀 Pull Request Oficial #27 hacia `feature/DS-Semana3-Nvidia`:**
   - Rama limpia, modular y con suite de 185 pruebas unitarias 100% en verde (353 tests integrados junto a PR #24 de DA).